# 🔄 Strava Token Refresher

This notebook checks if your stored Strava tokens are expired and refreshes them automatically using the refresh token.

The updated tokens will be saved back into `tokens_athletes.csv`, keeping your access valid for future API requests.

---

### ✅ What this notebook does:

1. Loads the tokens of all authorized athletes from the CSV file.
2. Checks which tokens are expired.
3. Sends a request to Strava to refresh them.
4. Updates the CSV with new access and refresh tokens.

> ⚠️ This process does **not require user interaction**. It can be scheduled to run periodically (e.g. once a day).


### 1. 📦 Load Required Libraries

In [1]:
import os
import time
import csv
import requests
import pandas as pd
from dotenv import load_dotenv
from pathlib import Path

### 2. ⚙️ Load Environment Variables

In [4]:
# Load environment variables from .env
load_dotenv()

CLIENT_ID = os.getenv("CLIENT_ID")
CLIENT_SECRET = os.getenv("CLIENT_SECRET")
TOKENS_PATH = r"C:\Users\dsgal\Documents\Activities\data\tokens_athletes.csv"

### 3. ⬆️ Load token CSV (Code)

In [5]:
# Load current tokens
df_tokens = pd.read_csv(TOKENS_PATH)
updated_tokens = []

### 4. 🔁 Refresh Access Token from Strava (Code)

In [6]:
for index, row in df_tokens.iterrows():
    name = row['nome']
    refresh_token = row['refresh_token']
    expires_at = int(row['expires_at'])

    if expires_at > int(time.time()):
        print(f"✅ Token still valid for {name}")
        updated_tokens.append(row)
        continue

    print(f"🔄 Refreshing token for {name}...")

    url = "https://www.strava.com/api/v3/oauth/token"
    payload = {
        'client_id': CLIENT_ID,
        'client_secret': CLIENT_SECRET,
        'grant_type': 'refresh_token',
        'refresh_token': refresh_token
    }

    response = requests.post(url, data=payload)

    if response.status_code == 200:
        new_data = response.json()
        updated_tokens.append({
            'nome': name,
            'athlete_id': row['athlete_id'],
            'access_token': new_data['access_token'],
            'refresh_token': new_data['refresh_token'],
            'expires_at': new_data['expires_at']
        })
        print(f"✅ Token refreshed for {name}")
    else:
        print(f"❌ Failed to refresh token for {name}: {response.status_code}")
        updated_tokens.append(row)

✅ Token still valid for Diego Galdino


### 5. 📥 Save Updated Token to CSV

In [7]:
df_updated = pd.DataFrame(updated_tokens)
df_updated.to_csv(TOKENS_PATH, index=False)
print(f"💾 Token file updated successfully: {TOKENS_PATH}")

💾 Token file updated successfully: C:\Users\dsgal\Documents\Activities\data\tokens_athletes.csv
